In [2]:
import gym 
import numpy as np
from collections import deque

In [3]:
#initialize the environment
env = gym.make('CartPole-v1')

#set Hyperparameters
alpha = 0.001  # Learning rate
gamma = 0.99   # Discount factor
epsilon = 1.0  # Exploration rate
epsilon_decay = 0.995
epsilon_min = 0.01
batch_size = 64
memory_size = 2000
episodes = 1000


In [4]:
#initialize replay buffer
memory = deque(maxlen=memory_size)

#get state and action sizes
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam   

def build_model(state_size, action_size):
    model = Sequential()
    model.add(Dense(24, input_dim=state_size, activation='relu'))
    model.add(Dense(24, activation='relu'))
    model.add(Dense(action_size, activation='linear'))
    model.compile(loss='mse', optimizer=Adam(learning_rate=alpha))
    return model

#build the Q-network and target network
q_network = build_model(state_size, action_size)
target_network = build_model(state_size, action_size)
target_network.set_weights(q_network.get_weights()) #initialize target network


/home/shobhit/.local/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1759748568.844023  119064 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2197 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


In [ ]:
def remember(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def replay(batch_size):
    minibatch = np.random.choice(len(memory), batch_size, replace=False)
    for _ in minibatch:
        state, action, reward, next_state, done = memory[_]
        target = q_network.predict(state)
        if done:
            target[0][action] = reward
        else:
            t = target_network.predict(next_state)
            target[0][action] = reward + gamma * np.amax(t)
        q_network.fit(state, target, epochs=1, verbose=0)

In [ ]:
#train the agent
#main training loop
for episode in range(episodes):
    state, info = env.reset()
    state = np.reshape(state, [1, state_size])
    total_reward = 0

    for time in range(500):
        # Epsilon-greedy action selection
        if np.random.rand() <= epsilon:
            action = np.random.choice(action_size)  # Explore
        else:
            q_values = q_network.predict(state, verbose=0)
            action = np.argmax(q_values[0])  # Exploit

        # Take action in the environment
        next_state, reward, done, truncated, info = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])

        # Penalize ending the episode early
        if done:
            reward = -10

        # Store experience and train
        remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

        if len(memory) > batch_size:
            replay(batch_size)

        if done:
            break

    # Decay epsilon after each episode
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

    # Update the target network weights every 10 episodes
    if episode % 10 == 0:
        target_network.set_weights(q_network.get_weights())

    print(f"Episode: {episode}, Score: {time}, Total Reward: {total_reward}, Epsilon: {epsilon:.4f}")


SyntaxError: 'break' outside loop (2350344059.py, line 21)